# E-Commerce Recommendation Engine
### Hybrid Content-Based + Collaborative Filtering Demo

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
from src.preprocessing import DataPreprocessor
from src.content_based import ContentBasedRecommender
from src.collaborative import CollaborativeFilteringModel
from src.hybrid import HybridRecommender

## 1. Load and Preprocess Data

In [ ]:
preprocessor = DataPreprocessor('../sample_dataset.csv')
df, products_df, ratings_df = preprocessor.run()
print(preprocessor.get_summary())
df.head()

## 2. Rating Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ratings_df['rating'], bins=9, color='steelblue', edgecolor='white')
ax.set_title('Rating Distribution')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Content-Based Recommendations

In [ ]:
content_model = ContentBasedRecommender(top_n=5)
content_model.fit(products_df)

recs = content_model.recommend('P001')
pd.DataFrame(recs)[['product_id','product_name','category','content_score','explanation']]

## 4. Collaborative Filtering

In [ ]:
collab_model = CollaborativeFilteringModel(n_factors=50, n_epochs=20)
collab_model.fit(ratings_df, evaluate=True)
print('Metrics:', collab_model.get_metrics())

collab_recs = collab_model.recommend('U001', rated_product_ids=['P001','P002','P003'], top_n=5)
pd.DataFrame(collab_recs)

## 5. Hybrid Recommendations

In [ ]:
hybrid = HybridRecommender(content_model, collab_model, products_df, ratings_df)

result = hybrid.recommend_for_user('U001', top_n=5)
print('Mode:', result['mode'])
pd.DataFrame(result['recommendations'])[['product_id','product_name','hybrid_score','confidence','explanation']]

## 6. Cold-Start User

In [ ]:
cold_result = hybrid.recommend_for_user('NEW_USER_999', top_n=5)
print('Mode:', cold_result['mode'])
pd.DataFrame(cold_result['recommendations'])[['product_id','product_name','category','hybrid_score']]

## 7. Product-Based Recommendations

In [ ]:
prod_result = hybrid.recommend_for_product('P003', top_n=5)
print('Product:', prod_result['product_name'])
pd.DataFrame(prod_result['recommendations'])[['product_id','product_name','category','content_score']]